In [1]:
import sys
sys.path.append('..')

from data.emission_factors import SCOPE1, SCOPE2_ELECTRICITY, BIOMETHANE
from data.sites import LACQ, FRENCH_GAS_SECTOR_AVG

site = LACQ
print(f"Running scenarios for: {site['name']}")

Running scenarios for: Lacq Gas Processing Site (illustrative)


In [2]:
def scenario_ppa(site, ef_electricity, ppa_fraction):
    """
    Model switching a fraction of grid electricity to renewable PPA.
    ppa_fraction: float between 0.0 (none) and 1.0 (full switch)
    """
    assert 0.0 <= ppa_fraction <= 1.0, 'ppa_fraction must be between 0 and 1'

    grid = site['grid']
    mwh = site['electricity_MWh']
    ef_grid = ef_electricity[grid]
    ef_ppa  = ef_electricity['renewable_ppa']

    # Weighted average emission factor after partial PPA adoption
    ef_new = (ppa_fraction * ef_ppa) + ((1 - ppa_fraction) * ef_grid)

    scope2_baseline = mwh * ef_grid / 1000
    scope2_new      = mwh * ef_new  / 1000
    reduction       = scope2_baseline - scope2_new

    return {
        'ppa_fraction':    ppa_fraction,
        'scope2_new_tCO2': round(scope2_new, 1),
        'reduction_tCO2':  round(reduction, 1),
        'reduction_pct':   round(reduction / scope2_baseline * 100, 1),
        'lever':           'Renewable electricity PPA',
    }

# Test at different adoption levels
for pct in [0.25, 0.50, 0.75, 1.00]:
    r = scenario_ppa(site, SCOPE2_ELECTRICITY, ppa_fraction=pct)
    print(f"PPA {pct*100:.0f}%:  -{r['reduction_tCO2']:>6,.0f} tCO2eq  "
          f"({r['reduction_pct']:.1f}% of Scope 2)")

PPA 25%:  -     3 tCO2eq  (20.5% of Scope 2)
PPA 50%:  -     6 tCO2eq  (40.9% of Scope 2)
PPA 75%:  -     8 tCO2eq  (61.4% of Scope 2)
PPA 100%:  -    11 tCO2eq  (81.8% of Scope 2)


In [3]:
def scenario_biomethane(site, ef_scope1, ef_biomethane,
                        bio_fraction, feedstock='agricultural_waste'):
    """
    Replace a fraction of natural gas with biomethane.
    bio_fraction: 0.0 to 1.0 of natural gas volume replaced.
    feedstock: key from BIOMETHANE dictionary in emission_factors.py
    """
    assert 0.0 <= bio_fraction <= 1.0, 'bio_fraction must be between 0 and 1'
    assert feedstock in ef_biomethane, f'Unknown feedstock: {feedstock}'

    ng_mwh       = site.get('natural_gas_MWh', 0)
    ng_remaining = ng_mwh * (1 - bio_fraction)   # still natural gas
    bio_mwh      = ng_mwh * bio_fraction          # switched to biomethane

    scope1_ng_new  = ng_remaining * ef_scope1['natural_gas'] 
    scope1_bio     = bio_mwh * ef_biomethane[feedstock]
    scope1_oil     = site.get('fuel_oil_MWh', 0) * ef_scope1['fuel_oil']

    scope1_new     = scope1_ng_new + scope1_bio + scope1_oil
    scope1_baseline = (ng_mwh * ef_scope1['natural_gas'] +
                       site.get('fuel_oil_MWh', 0) * ef_scope1['fuel_oil'])
    reduction      = scope1_baseline - scope1_new

    return {
        'feedstock':        feedstock,
        'bio_fraction':     bio_fraction,
        'scope1_new_tCO2':  round(scope1_new, 1),
        'reduction_tCO2':   round(reduction, 1),
        'reduction_pct':    round(reduction / scope1_baseline * 100, 1),
        'lever':            f'Biomethane ({feedstock})',
    }

# Compare all feedstocks at 30% substitution
print(f"{'Feedstock':<25} {'Reduction tCO2eq':>16} {'% of Scope 1':>13}")
print('-' * 56)
for feedstock in BIOMETHANE.keys():
    r = scenario_biomethane(site, SCOPE1, BIOMETHANE,
                            bio_fraction=0.30, feedstock=feedstock)
    print(f"{feedstock:<25} {-r['reduction_tCO2']:>16,.0f} {r['reduction_pct']:>12.1f}%")

Feedstock                 Reduction tCO2eq  % of Scope 1
--------------------------------------------------------
agricultural_waste                 -48,960         24.8%
food_waste                         -50,160         25.4%
sewage_sludge                      -47,760         24.1%
energy_crops                       -43,680         22.1%
landfill_gas                       -51,600         26.1%


In [12]:
def scenario_electrification(site, ef_scope1, ef_electricity,
                             elec_fraction, efficiency_ratio=0.90,
                             ppa_fraction=0.0):
    """
    Convert a fraction of gas-fired thermal processes to electricity.
    elec_fraction:   0.0 to 1.0 of natural gas processes electrified.
    efficiency_ratio: electric process efficiency vs gas (0.85-0.95).
    ppa_fraction:    fraction of new electricity demand from PPA.
    """
    assert 0.0 <= elec_fraction <= 1.0, 'elec_fraction must be between 0 and 1'

    ng_mwh        = site.get('natural_gas_MWh', 0)
    gas_displaced = ng_mwh * elec_fraction
    gas_remaining = ng_mwh * (1 - elec_fraction)

    # Electric processes need less energy than gas due to efficiency gains
    elec_added_mwh = gas_displaced * efficiency_ratio

    # Scope 1 decreases — less gas burned
    scope1_new = (gas_remaining * ef_scope1['natural_gas'] +
                  site.get('fuel_oil_MWh', 0) * ef_scope1['fuel_oil'])

    # Scope 2 increases — more electricity purchased
    elec_total_mwh = site['electricity_MWh'] + elec_added_mwh
    grid = site['grid']
    ef_blended = (ppa_fraction * ef_electricity['renewable_ppa'] +
                  (1 - ppa_fraction) * ef_electricity[grid])
    scope2_new = elec_total_mwh * ef_blended

    # Baselines
    scope1_baseline = (ng_mwh * ef_scope1['natural_gas'] +
                       site.get('fuel_oil_MWh', 0) * ef_scope1['fuel_oil'])
    scope2_baseline = site['electricity_MWh'] * ef_electricity[grid]
    baseline_total  = scope1_baseline + scope2_baseline
    new_total       = scope1_new + scope2_new

    return {
        'elec_fraction':  elec_fraction,
        'scope1_new':     round(scope1_new, 1),
        'scope2_new':     round(scope2_new, 1),
        'total_new_tCO2': round(new_total, 1),
        'net_reduction':  round(baseline_total - new_total, 1),
        'reduction_pct':  round((baseline_total - new_total) / baseline_total * 100, 1),
        'lever':          'Electrification of thermal processes',
    }

# Test at different electrification levels
print(f"{'Electrification':>16} {'Net reduction tCO2eq':>22} {'% of total':>11}")
print('-' * 52)
for frac in [0.10, 0.25, 0.50]:
    r = scenario_electrification(site, SCOPE1, SCOPE2_ELECTRICITY,
                                 elec_fraction=frac)
    print(f"{frac*100:>15.0f}%  {r['net_reduction']:>20,.0f}  {r['reduction_pct']:>10.1f}%")

 Electrification   Net reduction tCO2eq  % of total
----------------------------------------------------
             10%                14,200         6.7%
             25%                35,500        16.8%
             50%                71,000        33.6%


In [14]:
def scenario_ccs(scope1_residual_tCO2, capture_rate=0.90,
                 cost_eur_per_tCO2=100):
    """
    Apply carbon capture and storage to residual Scope 1 emissions.
    capture_rate:       fraction of Scope 1 captured (typically 0.85-0.95).
    cost_eur_per_tCO2:  European CCS cost range is 80-120 EUR/tCO2.
    """
    assert 0.0 <= capture_rate <= 1.0, 'capture_rate must be between 0 and 1'

    captured  = scope1_residual_tCO2 * capture_rate
    residual  = scope1_residual_tCO2 * (1 - capture_rate)
    cost_total = captured * cost_eur_per_tCO2

    return {
        'capture_rate':          capture_rate,
        'captured_tCO2':         round(captured, 1),
        'residual_tCO2':         round(residual, 1),
        'cost_EUR':              round(cost_total, 0),
        'cost_per_tCO2_avoided': cost_eur_per_tCO2,
        'lever':                 'CCS on residual Scope 1',
    }

# Apply CCS after other levers have reduced Scope 1
# Use biomethane 40% result as the residual Scope 1 input
bio_40 = scenario_biomethane(site, SCOPE1, BIOMETHANE,
                             bio_fraction=0.40,
                             feedstock='agricultural_waste')

ccs = scenario_ccs(bio_40['scope1_new_tCO2'], capture_rate=0.90)

print(f"Scope 1 after 40% biomethane:  {bio_40['scope1_new_tCO2']:>10,.0f} tCO2eq")
print(f"Captured by CCS:               {ccs['captured_tCO2']:>10,.0f} tCO2eq")
print(f"Residual after CCS:            {ccs['residual_tCO2']:>10,.0f} tCO2eq")
print(f"Cost of CCS:                   {ccs['cost_EUR']:>10,.0f} EUR")

Scope 1 after 40% biomethane:     132,520 tCO2eq
Captured by CCS:                  119,268 tCO2eq
Residual after CCS:                13,252 tCO2eq
Cost of CCS:                   11,926,800 EUR
